# FULL-SYN — chạy thử M5 trên task 2

[Phân tích round 1](../../docs/project/full-syn-r1-analysis-2026-10-29.md), mục 3.6 và 7. Giả thuyết: model nhầm các SKU
anh em vì backbone và encoder bị khoá sau FSA, mà FSA chỉ học SKU của task 1. M5 cho tầng cuối của backbone
(`stages_3`) và encoder học tiếp từ task 2 ở `lr_old`, kèm một loss giữ output của encoder gần với model thầy
(`--feature_kd`). Mọi thứ khác giữ như round 1, kể cả learning rate và số epoch, để chỉ đổi một biến.

Notebook này là **log của lần chạy thử**: FSA và PDP task 1 lấy nguyên từ round 1 (`seed_campaign`), chỉ train
**task 2**, rồi so với task 2 của round 1 trên val:

| Kết quả | Quyết định |
|---|---|
| cAcc-M2 tăng ≥ 3 điểm **và** AP50 các SKU task 1 giảm ≤ 1 điểm | `go`: làm lại prototype của SKU task 1, rồi chạy task 3–5 |
| cAcc-M2 tăng < 3 điểm | `stop`: giả thuyết sai, chuyển sang train ở 800px (hướng B) |
| Tăng nhưng SKU task 1 giảm > 1 điểm | `retry`: thử lại một lần với `feature_kd` lớn hơn |

Code chặn chạy quá task 2 và chặn chấm test, vì prototype của SKU task 1 phải được làm lại trước.
Bị ngắt giữa chừng thì chạy lại chính file này: bước đã xong được bỏ qua.

In [1]:
import json, os, sys
from pathlib import Path
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file())
sys.path.insert(0, str(REPO))
from autocheckout import full_syn
from autocheckout.io import load_json
from autocheckout.training_workflow import resolve
SETTINGS_PATH = resolve(os.environ.get('AUTOCHECKOUT_TRAINING_SETTINGS', 'data/training/settings_full_syn_m5_pilot.json'))
SETTINGS = load_json(SETTINGS_PATH)
PLAN = full_syn.prepare(SETTINGS)
print('Settings:', SETTINGS_PATH)
print('Campaign:', PLAN['root'], '| round', PLAN['round'], '| dry run' if PLAN['dry_run'] else '| real run')
print('Synthetic config sha256:', PLAN['release_config_sha256'])
print('Train images per task:', PLAN['train_images'])
print('PDP:', PLAN['pdp_experiment'], '| FSA và task 1 lấy từ:', PLAN['seed_root'], PLAN['seed'])
print(json.dumps({k: PLAN[k] for k in ('resolution', 'batch_size', 'effective_batch', 'fsa_batch_size', 'num_workers', 'tf32',
                                         'fsa_epochs', 'epochs', 'audit_images')}, indent=1))

Settings: /root/AutoCheckout-CL/data/training/full-syn-m5-pilot/notebook_executions/06_full_syn_m5_pilot/d5c62ccb-ede0-41de-83a8-7fc4c4dea0f6/settings.json
Campaign: /root/AutoCheckout-CL/data/training/full-syn-m5-pilot | round 1 | real run
Synthetic config sha256: cbd29ff0b63eb8aa93fdc48aef6d8eef0ca7116ac7cdc67bc3531fa6c5896eca
Train images per task: [21752, 12534, 12129, 11850, 13377]
PDP: EXP-FULL-SYN-M5 | FSA và task 1 lấy từ: /root/AutoCheckout-CL/data/training/full-syn-r1 {'campaign_id': 'full-syn-r1', 'round': 1, 'dry_run': False, 'task1_md5': '2a5fd7564b32a1663541569d0ec4c098'}
{
 "resolution": 640,
 "batch_size": 8,
 "effective_batch": 8,
 "fsa_batch_size": 4,
 "num_workers": 8,
 "tf32": true,
 "fsa_epochs": 8,
 "epochs": 16,
 "audit_images": 1000
}


In [2]:
import platform, subprocess, torch
from autocheckout.runinfo import git_state
git = git_state(REPO)
print('git', git.get('commit'), '| dirty' if git.get('dirty') else '| clean')
print('Python', platform.python_version(), '| torch', torch.__version__, '| CUDA', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'],
                     capture_output=True, text=True).stdout if torch.cuda.is_available() else '')

git 292afdc8a79f8372faa0971a465b1988b6717778 | dirty
Python 3.10.22 | torch 2.2.2+cu121 | CUDA 12.1
GPU: NVIDIA GeForce RTX 4090
name, memory.total [MiB], driver_version
NVIDIA GeForce RTX 4090, 24564 MiB, 565.77



## FSA và task 1 từ round 1

Link `runs/EXP-FULL-SYN-FSA` và `task_1`, chép báo cáo `fsa`/`task1` và policy đếm của stage 1. Kiểm md5 của
`task_1/task_final.pth` so với lúc chuẩn bị campaign.

In [3]:
SEED = full_syn.stage_seed(PLAN)
print(json.dumps({stage: {k: SEED[stage][k] for k in ('cAcc_m2', 'AP50', 'gate')} for stage in SEED}, indent=1))

{
 "fsa": {
  "cAcc_m2": 0.7765192376842862,
  "AP50": {
   "mAP_C": 0.9241394037628083,
   "mAP_P": null,
   "mAP_A": 0.9241394037628083
  },
  "gate": {
   "name": "D18 FSA early check",
   "decision": "go",
   "reason": "val cAcc-M2 0.7765 >= 0.528",
   "enforced": true,
   "value": 0.7765192376842862,
   "threshold": 0.528
  }
 },
 "task1": {
  "cAcc_m2": 0.7937792161093132,
  "AP50": {
   "mAP_C": 0.9338016073274583,
   "mAP_P": null,
   "mAP_A": 0.9338016073274583
  },
  "gate": {
   "name": "D17 task 1",
   "decision": "go",
   "reason": "cAcc-M2 0.7938 >= 0.728",
   "enforced": true,
   "colour": "green",
   "value": 0.7937792161093132,
   "green": 0.728,
   "red": 0.578,
   "round": 1
  }
 }
}


## Task 2 với M5

Trước khi train: audit nhãn giả của model task 1 (như round 1, cổng precision ≥ 0,5). Sau khi train: chấm val.

In [4]:
TASKS = full_syn.stage_tasks(PLAN, last=2)
NEW = TASKS['tasks'][2]
print(f"cAcc M2 {NEW['cAcc_m2']:.4f} | M1 {NEW['cAcc_m1_groups']:.4f} (nhóm) | AP50 {NEW['AP50']} | {NEW['train_hours']} h")
print('Nhầm lẫn nhiều nhất:', NEW['confusions_top20'][:10])

2026-10-30T07:26:39+07:00 START bash -ec 'source "$CONFIG"; cd "$REPO/pdp"; exec "$PYTHON" ppg_audit.py "${ARGS[@]}" "$@"' ppg_audit --output_dir /root/AutoCheckout-CL/data/training/full-syn-m5-pilot/runs/EXP-FULL-SYN-M5 --audit_task 2 --audit_images 1000 --prev_ckpt /root/AutoCheckout-CL/data/training/full-syn-m5-pilot/runs/EXP-FULL-SYN-M5/task_1/task_final.pth --task_ann_dir /root/AutoCheckout-CL/data/processed/rpc_synth_v2_v1/tasks/synth_v2 -> /root/AutoCheckout-CL/data/training/full-syn-m5-pilot/attempts/task2/f670d003-dade-4d31-b9ca-26cc846f4b07/ppg_audit_task2_runner.log
/venv/pdp/lib/python3.10/site-packages/lightning_utilities/core/imports.py:14: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
loading annotations into memory...
Done (t=1.71s)
creating index...
inde

## So với task 2 của round 1

In [5]:
R1 = load_json(Path(PLAN['seed_root']) / 'stages/tasks.json')['tasks']['2']
decision, reason = full_syn.pilot_verdict(NEW, R1)
print(decision.upper(), '-', reason)
r1_pairs = {(c['true'], c['predicted']): c['count'] for c in R1['confusions_top20']}
for c in NEW['confusions_top20'][:10]:
    print(f"  {c['true']:>3} -> {c['predicted']:>3}: {c['count']:>4} (round 1: {r1_pairs.get((c['true'], c['predicted']), '-')})")

STOP - cAcc-M2 0.7364 -> 0.7485 (+0.0121), AP50 task-1 classes 0.9496 -> 0.9473 (-0.0023): no gain, the frozen-feature hypothesis fails; next is 800 px (direction B)
   41 ->  59:   19 (round 1: 32)
   84 ->  85:   10 (round 1: 9)
  101 ->  11:    5 (round 1: 4)
   98 -> 111:    5 (round 1: 2)
  116 ->  66:    3 (round 1: 3)
   73 ->  72:    3 (round 1: 5)
  117 ->  66:    2 (round 1: -)
   64 ->  63:    2 (round 1: 2)
   69 ->  70:    2 (round 1: -)
   31 ->  87:    2 (round 1: -)
